In [ ]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

Review should focus on the behavior encoded in code cells, not on notebook metadata churn. This notebook keeps that review loop small: run fast.ai style hints when desired, and print nbdev-style code diffs when comparing notebooks.

Review tools draw a line between source changes and notebook noise. `style_check` is useful before publishing exported code because it combines fast.ai style hints with notebook hygiene reports, while `diff_nb` is useful during agent edits because it ignores outputs and metadata unless metadata is the only thing that changed.


### Production contract

Review tools are production core. Validation must fail on missing or stale nbskill metadata, `style_check` must combine capped style output with notebook hygiene diagnostics, and `diff_nb` must show code-cell behavior changes without raw notebook metadata noise.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import subprocess
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell, read_nb, write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.review import code_source

In [ ]:
print("code cell:", code_source(mk_cell("answer = 42", cell_type="code")))
print("markdown cell:", code_source(mk_cell("Some docs", cell_type="markdown")))

In [ ]:
#| export
import ast
import glob
import json
import os
import re
import subprocess
import sys
from collections import Counter
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.nbio import read_nb
from fastcore.script import Param, call_parse
from nbdev.diff import nbs_pair, source_diff

from nbskill.foundation import (
    empty_failure_map, failure_map_path, load_failure_map, cell_class_names,
    cell_source, cli_error, cli_return, exported_py_path, file_hash,
    is_export_directive, is_exported_code_cell, none_if_string, tracked_call,
)
from nbskill.knowledge import knowledge_style_problems

### Style feedback as a tool

`style_check` wraps the fast.ai style checker so it can be called from the CLI or MCP without each caller rebuilding command arguments or handling strict mode.

In [ ]:
#| export
skip_style_paths = "_proc __pycache__ src assets examples tests archive".split(" ")


In [ ]:

#| export
_LARGE_CODE_CELL_LINE_LIMIT = 30
_LARGE_MARKDOWN_CELL_LINE_LIMIT = 20


In [ ]:
#| export
_LARGE_CELL_FUNCTION_LIMIT = 2


In [ ]:
#| export
_LARGE_GENERATED_PY_LINE_LIMIT = 1000


In [ ]:
#| export
def _style_skip_paths(skip_path=None):
    paths = list(skip_style_paths)
    if skip_path is None: return paths
    if isinstance(skip_path, (list, tuple, set)): paths += [str(item) for item in skip_path]
    else: paths.append(str(skip_path))
    return paths


In [ ]:
#| export
def _style_root_is_skipped(path, skip_paths):
    parts = Path("." if path is None else path).expanduser().parts
    return any(part in skip_paths for part in parts)


In [ ]:
#| export
def _style_check_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["style_check", path]
    if skip_folder_re: argv += ["--skip-path-re", str(skip_folder_re)]
    for item in _style_skip_paths(skip_path): argv += ["--skip-path", item]
    return argv


In [ ]:
#| export
def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts


In [ ]:
#| export
def _notebook_paths(path="."):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file() and pth.suffix == ".ipynb":
        candidates = [pth]
    else:
        candidates = []
    return sorted({candidate for candidate in candidates if _is_notebook_path(candidate)})


In [ ]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not is_export_directive(line))


In [ ]:
#| export
def _parse_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return None


In [ ]:
#| export
def _top_level_function_count(tree):
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in tree.body)


In [ ]:
#| export
def _assert_count(tree):
    return sum(isinstance(node, ast.Assert) for node in ast.walk(tree))


In [ ]:
#| export
def _test_function_count(tree):
    return sum(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body
    )


In [ ]:
#| export
def _public_functions(tree):
    return [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and not node.name.startswith("_")
        and not node.name.startswith("test_")
    ]


In [ ]:
#| export
def _docstring_line_count(node):
    doc = ast.get_docstring(node, clean=False)
    if doc is None: return 0
    return len(doc.splitlines())


In [ ]:
#| export
def _public_function_docstring_problems(nb_path, cell, tree):
    if not is_exported_code_cell(cell): return []
    problems = []
    for node in _public_functions(tree):
        line_count = _docstring_line_count(node)
        if line_count == 1: continue
        if line_count == 0:
            detail = f"public function {node.name!r} needs a one-line docstring for nb_overview"
        else:
            detail = f"public function {node.name!r} has {line_count} docstring lines; keep it to one line for nb_overview"
        problems.append(_problem(
            "public-function-docstring", nb_path, cell, detail,
            symbol=node.name, line=getattr(node, "lineno", None), docstring_lines=line_count,
        ))
    return problems


In [ ]:
#| export
def _import_keys(tree):
    keys = []
    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".", 1)[0]
                keys.append(f"import {alias.name} as {local}")
        elif isinstance(node, ast.ImportFrom):
            module = "." * node.level + (node.module or "")
            for alias in node.names:
                local = alias.asname or alias.name
                keys.append(f"from {module} import {alias.name} as {local}")
    return keys


In [ ]:
#| export
def _cell_content_line_count(cell):
    source = cell_source(cell)
    if getattr(cell, "cell_type", None) == "code": source = _source_without_directives(source)
    return len([line for line in source.splitlines() if line.strip()])


In [ ]:

#| export
def _single_top_level_function_cell(tree):
    return len(tree.body) == 1 and isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef))


def _large_cell_problems(nb_path, cell, function_limit=_LARGE_CELL_FUNCTION_LIMIT):
    problems = []
    line_count = _cell_content_line_count(cell)
    cell_type = getattr(cell, "cell_type", "cell")
    tree = _parse_code_cell(cell)
    code_cell = cell_type == "code" and tree is not None
    line_limit = _LARGE_CODE_CELL_LINE_LIMIT if code_cell else _LARGE_MARKDOWN_CELL_LINE_LIMIT
    allow_long_cell = code_cell and _single_top_level_function_cell(tree)
    if line_count > line_limit and not allow_long_cell:
        detail = f"{line_count} {cell_type} content lines; keep one idea per cell"
        problems.append(_problem("large-cell", nb_path, cell, detail, line_count=line_count))
    if tree is None: return problems
    function_count = _top_level_function_count(tree)
    if function_count > function_limit:
        detail = f"{function_count} top-level functions; split into one-idea cells"
        problems.append(_problem("large-cell", nb_path, cell, detail, function_count=function_count))
    return problems


In [ ]:
#| export
def _generated_py_line_count(path):
    try: return len(Path(path).read_text(encoding="utf-8", errors="ignore").splitlines())
    except OSError: return 0


In [ ]:
#| export
def _generated_py_size_problem(nb_path, nb, line_limit=_LARGE_GENERATED_PY_LINE_LIMIT):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None or not py_path.exists(): return None
    line_count = _generated_py_line_count(py_path)
    if line_count <= line_limit: return None
    detail = f"{line_count} generated Python lines; split this notebook/module with the split tool"
    return _problem("large-generated-py", nb_path, detail=detail, exported_py_path=str(py_path), line_count=line_count)


In [ ]:
#| export
def _problem(code, path, cell=None, detail="", severity="warning", source="nbskill", **kwargs):
    problem = {
        "code": code,
        "path": str(path),
        "severity": severity,
        "source": source,
        "detail": detail,
    }
    if cell is not None: problem["cell_id"] = getattr(cell, "id", "")
    problem.update({key: value for key, value in kwargs.items() if value is not None})
    return problem


In [ ]:
#| export
def _format_problem(problem):
    cell = f" id={problem['cell_id']}" if problem.get("cell_id") else ""
    line = f" line={problem['line']}" if problem.get("line") else ""
    fields = []
    for key in (
        "scope", "symbol", "import_key", "cells", "semantic_types", "confidence", "hint",
        "exported_py_path", "line_count", "function_count", "docstring_lines",
        "regex", "note", "match",
    ):
        if key in problem:
            value = problem[key]
            if isinstance(value, list): value = ", ".join(map(str, value))
            fields.append(f"{key}={value!r}" if key in {"symbol", "import_key", "exported_py_path", "regex", "note", "match"} else f"{key}={value}")
    suffix = " ".join(fields + ([problem.get("detail", "")] if problem.get("detail") else []))
    return f"- {problem['code']}: {problem['path']}{cell}{line} {suffix}".rstrip()


In [ ]:
#| export
def _nbskill_info(obj):
    meta = getattr(obj, "metadata", {}) or {}
    return meta.get("nbskill") if isinstance(meta, dict) else None


In [ ]:
#| export
def _validation_problem(code, path, cell=None, detail="", **kwargs):
    return _problem(code, path, cell, detail=detail, severity="error", source="nbskill-validation", **kwargs)


In [ ]:
#| export
def _cell_metadata_validation_problems(nb_path, cell):
    info = _nbskill_info(cell)
    if not isinstance(info, dict):
        return [_validation_problem("missing-cell-nbskill-metadata", nb_path, cell)]
    problems = []
    expected_type = getattr(cell, "cell_type", None)
    if "cell_type" not in info:
        problems.append(_validation_problem("missing-cell-type", nb_path, cell))
    elif info.get("cell_type") != expected_type:
        problems.append(_validation_problem("cell-type-mismatch", nb_path, cell, f"expected {expected_type!r}, stored {info.get('cell_type')!r}"))
    semantic_types = info.get("semantic_types")
    if not isinstance(semantic_types, list) or not semantic_types:
        problems.append(_validation_problem("missing-cell-semantic-types", nb_path, cell))
    return problems


In [ ]:
#| export
def _notebook_export_hash_problems(nb_path, nb):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return []
    info = _nbskill_info(nb)
    if not py_path.exists():
        return [_validation_problem("exported-py-missing", nb_path, detail=f"missing {py_path}", exported_py_path=str(py_path))]
    if not isinstance(info, dict) or not info.get("exported_py_hash"):
        return [_validation_problem("missing-exported-py-hash", nb_path, exported_py_path=str(py_path))]
    actual = file_hash(py_path)
    stored = info.get("exported_py_hash")
    if stored != actual:
        detail = f"expected {actual[:12]}, stored {str(stored)[:12]}"
        return [_validation_problem("exported-py-hash-mismatch", nb_path, detail=detail, exported_py_path=str(py_path))]
    return []


In [ ]:
#| export
def notebook_validation_problems(path="."):
    "Return invalid nbskill metadata problems for notebooks under `path`."
    problems = []
    for nb_path in _notebook_paths(path):
        if _style_root_is_skipped(nb_path, skip_style_paths): continue
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_notebook_export_hash_problems(nb_path, nb))
        for cell in nb.cells: problems.extend(_cell_metadata_validation_problems(nb_path, cell))
    return problems


In [ ]:
#| export
def _notebook_size_problems_for_nb(nb_path, nb):
    problems = []
    for cell in nb.cells: problems.extend(_large_cell_problems(nb_path, cell))
    generated_problem = _generated_py_size_problem(nb_path, nb)
    if generated_problem: problems.append(generated_problem)
    return problems


In [ ]:
#| export
def notebook_size_problems(path="."):
    "Return size warnings for notebook cells and generated Python files."
    problems = []
    for nb_path in _notebook_paths(path):
        if _style_root_is_skipped(nb_path, skip_style_paths): continue
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_notebook_size_problems_for_nb(nb_path, nb))
    return problems


In [ ]:
#| export
def _notebook_style_problems(path="."):
    problems = notebook_validation_problems(path)
    duplicate_imports = {}
    for nb_path in _notebook_paths(path):
        if _style_root_is_skipped(nb_path, skip_style_paths): continue
        try:
            nb = read_nb(nb_path)
        except FileNotFoundError:
            continue
        problems.extend(_notebook_size_problems_for_nb(nb_path, nb))
        imports_by_scope = {"exported": {}, "internal": {}}
        for cell in nb.cells:
            classes = cell_class_names(cell)
            if "unclean_cell" in classes:
                visible = [name for name in classes if name != "unclean_cell"]
                problems.append(_problem("unclean-cell", nb_path, cell, semantic_types=visible))
            tree = _parse_code_cell(cell)
            if tree is None: continue
            problems.extend(_public_function_docstring_problems(nb_path, cell, tree))
            if "test_cell" in classes:
                problem_count = _assert_count(tree) + _test_function_count(tree)
                if problem_count > 3:
                    problems.append(_problem("multi-problem-test", nb_path, cell, f"{problem_count} asserts/test functions; split into one-problem-at-a-time cells"))
            scope = "exported" if is_exported_code_cell(cell) else "internal"
            for key in _import_keys(tree):
                imports_by_scope[scope].setdefault(key, []).append(getattr(cell, "id", ""))
        for scope, imports in imports_by_scope.items():
            for key, ids in imports.items():
                if len(ids) > 1:
                    duplicate_imports.setdefault(str(nb_path), []).append((scope, key, ids))
    for nb_path, items in duplicate_imports.items():
        for scope, key, ids in items:
            problems.append(_problem("duplicate-import", nb_path, scope=scope, import_key=key, cells=ids))
    problems.extend(
        problem for problem in knowledge_style_problems(path)
        if not _style_root_is_skipped(problem["path"], skip_style_paths)
    )
    if _notebook_paths(path):
        from nbskill.graph import notebook_order_problems
        problems.extend(
            problem for problem in notebook_order_problems(path)
            if not _style_root_is_skipped(problem["path"], skip_style_paths)
        )
    return problems


In [ ]:
#| export
def _notebook_style_problem_lines(path="."):
    return [_format_problem(problem) for problem in _notebook_style_problems(path)]


In [ ]:
#| export
def _format_notebook_style_report(path="."):
    lines = _notebook_style_problem_lines(path)
    if not lines: return "Notebook style report: no notebook hygiene problems found."
    return "\n".join(["Notebook style report:", *lines])


In [ ]:
#| export
def _format_count_group(title, counts):
    if not counts: return [f"{title}: none"]
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [f"{title}: " + ", ".join(f"{tool}={count}" for tool, count in ordered)]


In [ ]:
#| export
def _format_failure_event(event):
    kind = event.get("kind", "event")
    tool = event.get("tool", "unknown")
    path = event.get("path")
    detail = event.get("summary") or event.get("error") or ",".join(event.get("reasons", []))
    location = f" path={path}" if path else ""
    return f"- {kind}: {tool}{location} {detail}".rstrip()


In [ ]:
#| export
def _global_usage_summary_data():
    path = failure_map_path()
    data = load_failure_map(path) if path.exists() else empty_failure_map()
    problems = [event for event in data.get("events", []) if event.get("kind") in {"failure", "friction"}][-5:]
    return {
        "path": str(path),
        "exists": path.exists(),
        "counts": data.get("counts", {}),
        "recent_problems": problems,
    }


In [ ]:
#| export
def _format_global_usage_summary(data=None):
    data = data or _global_usage_summary_data()
    if not data["exists"]: return f"Global nbskill usage: no records at {data['path']}"
    counts = data.get("counts", {})
    lines = [f"Global nbskill usage: {data['path']}"]
    lines += _format_count_group("usage", counts.get("usage", {}))
    lines += _format_count_group("failures", counts.get("failures", {}))
    lines += _format_count_group("friction", counts.get("friction", {}))
    problems = data.get("recent_problems", [])
    if problems:
        lines.append("recent problems:")
        lines.extend(_format_failure_event(event) for event in problems)
    else:
        lines.append("recent problems: none")
    return "\n".join(lines)


In [ ]:
#| export
def reset_global_usage_summary():
    path = failure_map_path()
    try: path.unlink()
    except FileNotFoundError: pass
    except OSError: path.write_text(json.dumps(empty_failure_map(), indent=2, sort_keys=True), encoding="utf-8")


In [ ]:
#| export
_CHKSTYLE_RE = re.compile(r"^# (?P<path>.*?):cell\[(?P<cell>[^\]]+)\]:(?P<line>\d+): (?P<detail>.*)$")


In [ ]:
#| export
def _cap_text(text, max_output_chars=12000):
    text = text or ""
    if max_output_chars is None or len(text) <= max_output_chars:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - max_output_chars
    return {
        "text": f"{text[:max_output_chars].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


In [ ]:
#| export
def _chkstyle_diagnostics(text, max_diagnostics=200):
    diagnostics = []
    for line in (text or "").splitlines():
        match = _CHKSTYLE_RE.match(line)
        if not match: continue
        detail = match.group("detail")
        hint = None
        if "(hint:" in detail:
            detail, hint = detail.split("(hint:", 1)
            hint = hint.rstrip(")").strip()
        diagnostics.append({
            "source": "chkstyle",
            "code": detail.strip().split(" (", 1)[0],
            "path": match.group("path"),
            "cell_id": match.group("cell"),
            "line": int(match.group("line")),
            "severity": "hint",
            "detail": detail.strip(),
            "hint": hint,
        })
        if max_diagnostics and len(diagnostics) >= max_diagnostics: break
    return diagnostics


In [ ]:
#| export
def _problem_chart(diagnostics):
    def counts(key):
        return dict(sorted(Counter(str(item.get(key, "")) for item in diagnostics if item.get(key)).items()))
    return {
        "by_code": counts("code"),
        "by_severity": counts("severity"),
        "by_path": counts("path"),
        "by_source": counts("source"),
    }


In [ ]:
#| export
def _fix_suggestions(diagnostics):
    fixes = []
    for item in diagnostics:
        if item.get("code") == "duplicate-import":
            fixes.append({
                "code": "duplicate-import",
                "path": item.get("path"),
                "cells": item.get("cells", []),
                "description": f"Remove repeated import {item.get('import_key')} from later cells after checking scope.",
                "automatic": False,
            })
    return fixes


In [ ]:
#| export
def style_report(
    path: str = ".",  # File or folder to check
    chkstyle: dict | None = None,  # Captured chkstyle result to include
    max_output_chars: int = 12000,  # Maximum raw chkstyle text to keep in report
    max_diagnostics: int = 200,  # Maximum parsed chkstyle diagnostics
    changed_only: bool = False,  # Report only diagnostics for changed code cells
    ref_a: str | None = "HEAD",  # First git ref for changed_only filtering
    ref_b: str | None = None,  # Second git ref; defaults to working tree
):
    "Return structured style diagnostics, problem chart, and global nbskill usage data."
    notebook_problems = _notebook_style_problems(path)
    usage = _global_usage_summary_data()
    chkstyle = chkstyle or {"status": 0, "output": ""}
    capped = _cap_text(chkstyle.get("output", ""), max_output_chars=max_output_chars)
    diagnostics = [
        *(_chkstyle_diagnostics(chkstyle.get("output", ""), max_diagnostics=max_diagnostics)),
        *notebook_problems,
    ]
    changed_cell_ids = _changed_code_cell_ids(path, ref_a=ref_a, ref_b=ref_b) if changed_only else None
    if changed_cell_ids is not None:
        diagnostics = [item for item in diagnostics if item.get("cell_id") in changed_cell_ids]
        notebook_problems = [item for item in notebook_problems if item.get("cell_id") in changed_cell_ids]
    notebook_text = _format_notebook_style_report(path)
    usage_text = _format_global_usage_summary(usage)
    chkstyle_text = capped["text"].strip()
    if changed_cell_ids is not None:
        text = _format_style_delta_report(path, diagnostics, changed_cell_ids, usage_text)
    else:
        text = "\n\n".join(chunk for chunk in [chkstyle_text, notebook_text, usage_text] if chunk)
    return {
        "path": str(path),
        "summary": {
            "notebook_problem_count": len(notebook_problems),
            "chkstyle_problem_count": len([item for item in diagnostics if item.get("source") == "chkstyle"]),
            "diagnostic_count": len(diagnostics),
            "recent_problem_count": len(usage.get("recent_problems", [])),
            "output_truncated": capped["truncated"],
            "output_chars": capped["chars"],
            "omitted_chars": capped["omitted_chars"],
            "changed_only": changed_only,
            "changed_cell_count": len(changed_cell_ids or []),
        },
        "diagnostics": diagnostics[:max_diagnostics] if max_diagnostics else diagnostics,
        "problem_chart": _problem_chart(diagnostics),
        "notebook_problems": notebook_problems,
        "global_usage": usage,
        "chkstyle": {"status": chkstyle.get("status", 0), **capped},
        "fixes": _fix_suggestions(diagnostics),
        "text": text,
    }


In [ ]:
report = style_report("nbs/data/test_nbskill.ipynb")
assert "summary" in report
assert "notebook_problems" in report
assert isinstance(report["notebook_problems"], list)
assert "Global nbskill usage:" in report["text"]

The structured report is meant for tools as much as people. `summary` gives a compact count of problems, `problem_chart` groups diagnostics by source and code, and `text` is the human-readable report that the CLI prints.

In [ ]:
sample_chkstyle = {
    "status": 1,
    "output": "# demo.ipynb:cell[abc123]:2: Missing whitespace (hint: add a blank line)",
}
sample_report = style_report(
    "nbs/data/test_nbskill.ipynb",
    chkstyle=sample_chkstyle,
    max_output_chars=200,
    max_diagnostics=5,
)
print(sample_report["summary"])
print(sample_report["problem_chart"]["by_source"])
print(sample_report["diagnostics"][0])

{'notebook_problem_count': 10, 'chkstyle_problem_count': 1, 'diagnostic_count': 11, 'recent_problem_count': 5, 'output_truncated': False, 'output_chars': 72, 'omitted_chars': 0, 'changed_only': False, 'changed_cell_count': 0}
{'chkstyle': 1, 'nbskill': 10}
{'source': 'chkstyle', 'code': 'Missing whitespace', 'path': 'demo.ipynb', 'cell_id': 'abc123', 'line': 2, 'severity': 'hint', 'detail': 'Missing whitespace', 'hint': 'add a blank line'}


In [ ]:
#| export
def run_style_check(path=".", skip_folder_re=None, skip_path=None, strict=False, max_output_chars=None):
    "Run chkstyle with nbskill's default skip paths and capped output."
    skip_paths = _style_skip_paths(skip_path)
    if _style_root_is_skipped(path, skip_paths):
        return {"status": 0, "output": "", "text": "", "truncated": False, "chars": 0, "omitted_chars": 0}
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        status = _chkstyle_main(_style_check_argv(path, skip_folder_re, skip_path))
    output = "\n".join(chunk.rstrip() for chunk in (out.getvalue(), err.getvalue()) if chunk)
    capped = _cap_text(output, max_output_chars=max_output_chars) if max_output_chars else {"text": output, "truncated": False, "chars": len(output), "omitted_chars": 0}
    return {"status": status, "output": output, **capped}

In [ ]:
argv = _style_check_argv(".")
assert "--skip-path" in argv
assert "_proc" in argv
assert "--skip-folder-re" not in argv
assert "--skip-path-re" in _style_check_argv(".", skip_folder_re=r"^_proc/")

proc_report = run_style_check("_proc/01_read.ipynb")
assert proc_report["status"] == 0
assert proc_report["output"] == ""
assert notebook_validation_problems("_proc/01_read.ipynb") == []
assert not any(_style_root_is_skipped(problem["path"], skip_style_paths) for problem in _notebook_style_problems("."))

In [ ]:
#| export
def _normalize_style_check_cli_aliases():
    aliases = {
        "--delete-after-output": "--delete_after_output",
        "--delete-after-outout": "--delete_after_outout",
    }
    sys.argv[:] = [aliases.get(arg, arg) for arg in sys.argv]


In [ ]:
#| export
_normalize_style_check_cli_aliases()


In [ ]:
#| export
@call_parse
@tracked_call
def style_check(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints or notebook hygiene problems are found
    delete_after_output: bool = False,  # Reset ~/.nbskill-errors.json after printing the global summary
    delete_after_outout: bool = False,  # Backward-compatible typo alias for delete_after_output
    max_output_chars: int = 12000,  # Cap printed chkstyle output
    max_diagnostics: int = 200,  # Cap returned diagnostics
    fix: bool = False,  # Show conservative fix suggestions
    dry_run: bool = True,  # Keep fix mode non-mutating by default
    changed_only: bool = False,  # Report only diagnostics for changed code cells
    ref_a: str | None = "HEAD",  # First git ref for changed_only filtering
    ref_b: str | None = None,  # Second git ref; defaults to working tree
):
    "Print capped fast.ai style hints, notebook hygiene warnings, and global tool usage."
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    report = style_report(path, chkstyle=chkstyle, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b)
    print(report["text"])
    if fix:
        print("\nFix suggestions:")
        fixes = report.get("fixes", [])
        if not fixes: print("- no deterministic fixes available")
        for item in fixes:
            mode = "would apply" if dry_run else "manual-review-required"
            print(f"- {mode}: {item['description']} ({item['path']})")
    if delete_after_output or delete_after_outout: reset_global_usage_summary()
    has_problems = bool(report["diagnostics"])
    if strict and (chkstyle["status"] or has_problems): raise SystemExit(chkstyle["status"] or 1)
    return cli_return(chkstyle["status"] or int(has_problems))


In [ ]:
#| export
@call_parse
@tracked_call
def validate_nbs(
    path: Param("Notebook file, folder, or glob to validate", str, opt=False, nargs="?") = "nbs",  # Notebook file, folder, or glob to validate
    strict: bool = True,  # Exit non-zero when invalid metadata is found
):
    "Validate nbskill metadata needed for safe notebook tools."
    problems = notebook_validation_problems(path)
    if problems:
        print("Notebook validation errors:")
        for problem in problems:
            print(_format_problem(problem))
        if strict: raise SystemExit(1)
    else:
        print("Notebook validation: no invalid nbskill metadata found.")
    return cli_return(int(bool(problems)))


In [ ]:
#| export
def code_source(cell):
    "Return source for code cells; ignore markdown and raw cells."
    return cell.source if cell.cell_type == "code" else None


`style_check` is the CLI-shaped wrapper around `style_report`. It captures fast.ai style output, appends notebook hygiene findings, prints the combined text report, and exits non-zero in strict mode when diagnostics are present.

In [ ]:
style_output = StringIO()
with redirect_stdout(style_output):
    style_check(
        "nbs/data/test_nbskill.ipynb",
        strict=False,
        max_output_chars=500,
        max_diagnostics=5,
    )
print("\n".join(style_output.getvalue().splitlines()[:6]))

In [ ]:
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import stamp_notebook_metadata, write_demo_notebook
from nbskill.review import notebook_validation_problems, validate_nbs

with write_demo_notebook("04_review_validate_valid.ipynb") as valid_path:
    valid_nb = stamp_notebook_metadata(new_nb([mk_cell("assert True", cell_type="code")]))
    _write_test_nb(valid_nb, valid_path)
    assert notebook_validation_problems(valid_path) == []
    validate_nbs(str(valid_path), strict=False)

In [ ]:
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import stamp_notebook_metadata, write_demo_notebook
from nbskill.review import notebook_validation_problems

with write_demo_notebook("04_review_validate_invalid.ipynb") as invalid_path:
    invalid_nb = stamp_notebook_metadata(new_nb([
        mk_cell("x = 1", cell_type="code"),
        mk_cell("Some docs", cell_type="markdown"),
        mk_cell("assert True", cell_type="code"),
    ]))
    invalid_nb.cells[1].metadata["nbskill"]["semantic_types"] = []
    invalid_nb.cells[2].metadata["nbskill"]["cell_type"] = "markdown"
    _write_test_nb(invalid_nb, invalid_path)
    codes = {problem["code"] for problem in notebook_validation_problems(invalid_path)}
    assert {"missing-cell-semantic-types", "cell-type-mismatch"} <= codes

In [ ]:
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import stamp_notebook_metadata, write_demo_notebook
from nbskill.review import notebook_validation_problems

with write_demo_notebook("04_review_export_hash.ipynb", base="nbs") as export_nb_path:
    export_nb = stamp_notebook_metadata(new_nb([mk_cell("#| default_exp sample_tool", cell_type="code")]))
    export_nb.metadata["nbskill"] = {"exported_py_hash": "bad"}
    _write_test_nb(export_nb, export_nb_path)
    codes = {problem["code"] for problem in notebook_validation_problems(export_nb_path)}
    assert "exported-py-hash-mismatch" in codes

`notebook_validation_problems` is narrower than `style_report`: it only checks the nbskill metadata that keeps notebook structure and exports reliable. `validate_nbs` prints the same failures for CLI use.

In [ ]:
with write_demo_notebook("04_review_validation_example.ipynb") as validation_path:
    validation_nb = new_nb([
        mk_cell("x = 1", cell_type="code"),
        mk_cell("Some docs", cell_type="markdown"),
    ])
    _write_test_nb(validation_nb, validation_path)
    problems = notebook_validation_problems(validation_path)
    print([problem["code"] for problem in problems[:4]])

### Code-cell diffs

Notebook diffs are noisy when metadata and outputs are included. `diff_nb` asks nbdev for code-cell source on each side of a comparison and prints only the added, changed, or deleted code blocks the caller requested.

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return f"No git repository found for {str(path)!r}."
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass --ref_a None to compare against the working tree."
    )


In [ ]:
#| export
def _git_root_rel(path):
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0: return None, None
    root = Path(root_cmd.stdout.strip())
    try: return root, path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return None, None


In [ ]:
#| export
def _notebook_json_at_ref(path, ref):
    path = Path(path)
    if ref is None:
        return json.loads(path.read_text(encoding="utf-8"))
    root, rel = _git_root_rel(path)
    if root is None: return None
    show = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    if show.returncode != 0: return None
    return json.loads(show.stdout)


In [ ]:
#| export
def _nbskill_metadata_by_cell(nb_json):
    cells = (nb_json or {}).get("cells", [])
    return {
        cell.get("id", str(idx)): (cell.get("metadata", {}) or {}).get("nbskill")
        for idx, cell in enumerate(cells)
    }


In [ ]:
#| export
def _nbskill_metadata_change_count(path, ref_a, ref_b):
    try:
        old = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_a))
        new = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_b))
    except (OSError, json.JSONDecodeError, TypeError):
        return 0
    keys = set(old) | set(new)
    return sum(1 for key in keys if old.get(key) != new.get(key) and (old.get(key) is not None or new.get(key) is not None))


In [ ]:
#| export
def _metadata_summary(count):
    if not count: return ""
    noun = "cell" if count == 1 else "cells"
    return f"Ignored nbskill metadata changes in {count} {noun}."


In [ ]:
#| export
def _changed_code_cell_ids(path, ref_a="HEAD", ref_b=None):
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception:
        return set()
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    return {cid for cid in set(old) | set(new) if old.get(cid) != new.get(cid)}


In [ ]:
#| export
def _diff_filter_ids(path, ref_b=None, cell_id=None, after_id=None):
    if not cell_id and not after_id: return None
    nb_json = _notebook_json_at_ref(path, none_if_string(ref_b))
    cells = (nb_json or {}).get("cells", [])
    ids = [cell.get("id", str(idx)) for idx, cell in enumerate(cells)]
    selected = set(ids)
    if cell_id: selected &= set(str(cell_id).replace(",", " ").split())
    if after_id:
        if after_id not in ids: return selected & set()
        selected &= set(ids[ids.index(after_id) + 1:])
    return selected


In [ ]:
#| export
def _format_style_delta_report(path, diagnostics, changed_cell_ids, usage_text):
    header = f"Style delta for changed code cells in {path}: {len(diagnostics)} diagnostic(s) across {len(changed_cell_ids)} changed cell(s)"
    if not diagnostics: return "\n\n".join([header, usage_text])
    lines = [header]
    for item in diagnostics:
        cell = f" id={item['cell_id']}" if item.get("cell_id") else ""
        line = f" line={item['line']}" if item.get("line") else ""
        detail = item.get("detail") or item.get("code", "")
        lines.append(f"- {item.get('source', 'nbskill')}: {item.get('path', path)}{cell}{line} {detail}".rstrip())
    return "\n\n".join(["\n".join(lines), usage_text])


In [ ]:
#| export
@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
    cell_id: str | None = None,  # Limit output to comma/space-separated cell ids
    after_id: str | None = None,  # Limit output to cells after this id in ref_b/the working tree
):
    "Print nbdev-style diffs for code cells only; summarize nbskill metadata-only changes."
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
        cli_error(msg)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception as exc:
        detail = str(exc)
        hint = (
            f"Could not diff {path!r} against {ref_a!r}. "
            "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
            "Commit the notebook first, or pass --ref_a None to compare against the working tree."
        )
        if detail: hint += f"\nUnderlying error: {detail}"
        cli_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    selected = _diff_filter_ids(path, ref_b=ref_b, cell_id=cell_id, after_id=after_id)
    if selected is not None: blocks = [(cid, diff) for cid, diff in blocks if cid in selected]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
    if text and metadata_summary: report = f"{text}\n\n{metadata_summary}"
    elif text: report = text
    elif metadata_summary: report = f"No code cell changes\n{metadata_summary}"
    else: report = "No code cell changes"
    print(report)
    return cli_return(report)


In [ ]:
with write_demo_notebook("04_review_no_git.ipynb") as path:
    write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    try:
        diff_nb(str(path))
    except (SystemExit, ValueError) as exc:
        if isinstance(exc, SystemExit): assert exc.code == 1
        else:
            msg = str(exc)
            assert "No git repository" in msg or "Could not find notebook" in msg


In [ ]:
def _committed_review_notebook(root):
    root.mkdir()
    path = root / "demo.ipynb"
    write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    subprocess.run(["git", "init"], cwd=root, check=True, capture_output=True)
    subprocess.run(["git", "add", "demo.ipynb"], cwd=root, check=True, capture_output=True)
    subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=root, check=True, capture_output=True)
    return path


In [ ]:
root = demo_path("04_review_git_metadata")
try:
    path = _committed_review_notebook(root)
    nb = read_nb(path)
    nb.cells[0].metadata["nbskill"] = {"cell_type": "code", "semantic_types": []}
    write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out):
        diff_nb(str(path))
    text = out.getvalue()
    assert "No code cell changes" in text
    assert "Ignored nbskill metadata changes in 1 cell" in text
finally:
    remove_demo_path(root)


In [ ]:
root = demo_path("04_review_git_filters")
try:
    path = _committed_review_notebook(root)
    nb = read_nb(path)
    first_id = nb.cells[0].id
    nb.cells[0].source = "x = 2"
    nb.cells.append(mk_cell("y = 3", cell_type="code"))
    write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out): diff_nb(str(path), cell_id=first_id)
    text = out.getvalue()
    assert "x = 2" in text and "y = 3" not in text
    out = StringIO()
    with redirect_stdout(out): diff_nb(str(path), after_id=first_id)
    text = out.getvalue()
    assert "y = 3" in text and "x = 2" not in text
    delta = style_report(str(path), changed_only=True)
    assert delta["summary"]["changed_only"] is True
    assert delta["summary"]["changed_cell_count"] == 2
finally:
    remove_demo_path(root)


In [ ]:
from contextlib import redirect_stdout


In [ ]:
from io import StringIO


In [ ]:
import os


In [ ]:
from fastcore.nbio import mk_cell, new_nb


In [ ]:
from fastcore.nbio import write_nb


In [ ]:
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook


In [ ]:
from nbskill.review import (
    _format_global_usage_summary, _format_notebook_style_report,
    notebook_size_problems, style_check,
)


In [ ]:
generated_py_path = None


In [ ]:
def _write_style_problem_notebook(path):
    long_source = "\n".join([f"x{i} = {i}" for i in range(31)])
    long_markdown = "\n".join([f"paragraph {i}" for i in range(21)])
    nb = new_nb([
        mk_cell(
            "#| export\n"
            "def a():\n"
            "    pass\n"
            "def b():\n"
            "    \"\"\"Describe b.\"\"\"\n"
            "    pass\n"
            "def c():\n"
            "    \"\"\"Line one.\n"
            "    Line two.\"\"\"\n"
            "    pass\n"
            "def _private():\n"
            "    pass",
            cell_type="code",
        ),
        mk_cell(long_source, cell_type="code"),
        mk_cell(long_markdown, cell_type="markdown"),
        mk_cell("assert 1 == 1\nassert 2 == 2\nassert 3 == 3\nassert 4 == 4", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("assert True\ndef _helper():\n    return 1", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("result = later_helper()", cell_type="code"),
        mk_cell("def later_helper():\n    return 1", cell_type="code"),
        mk_cell("def loader():\n    return MissingPath('x')", cell_type="code"),
    ])
    write_nb(nb, path)


In [ ]:
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    report = _format_notebook_style_report(path)
    assert "large-cell" in report
    assert "markdown content lines" in report
    assert "multi-problem-test" in report
    assert "unclean-cell" in report
    assert "scope=exported" in report
    assert "scope=internal" in report
    assert "cell-order" in report
    assert "missing-import" in report
    assert "public-function-docstring" in report
    assert "symbol='a'" in report
    assert "symbol='c'" in report
    assert "symbol='b'" not in report
    assert "symbol='_private'" not in report


In [ ]:
with write_demo_notebook("04_review_generated.ipynb") as generated_path:
    generated_nb = new_nb([mk_cell("#| default_exp big_review_demo", cell_type="code")])
    write_nb(generated_nb, generated_path)
    generated_py_path = exported_py_path(generated_path, generated_nb)
    try:
        generated_py_path.parent.mkdir(parents=True, exist_ok=True)
        generated_py_path.write_text("\n".join(f"x{i} = {i}" for i in range(1001)), encoding="utf-8")
        size_codes = {problem["code"] for problem in notebook_size_problems(generated_path)}
        assert "large-generated-py" in size_codes
    finally:
        if generated_py_path is not None: remove_demo_path(generated_py_path)


In [ ]:
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    custom_map = demo_path("04_review_errors.json")
    old_map = os.environ.get("NBSKILL_FAILURE_MAP")
    try:
        os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
        with redirect_stdout(StringIO()):
            style_check(str(path), delete_after_output=True)
        assert "usage:" in _format_global_usage_summary()
        assert not custom_map.exists()
    finally:
        if old_map is None: os.environ.pop("NBSKILL_FAILURE_MAP", None)
        else: os.environ["NBSKILL_FAILURE_MAP"] = old_map


In [ ]:
with write_demo_notebook("04_review_style.ipynb") as path:
    _write_style_problem_notebook(path)
    try:
        with redirect_stdout(StringIO()): style_check(str(path), strict=True)
    except SystemExit as exc:
        assert exc.code
    else:
        raise AssertionError("strict style_check should exit for notebook hygiene problems")


In [ ]:
assert code_source(mk_cell("plain docs", cell_type="markdown")) is None
assert code_source(mk_cell("answer = 42", cell_type="code")) == "answer = 42"

`diff_nb` deliberately compares code-cell source only. Markdown edits and notebook metadata churn stay out of the main diff so reviewers can see the executable behavior that changed.

In [ ]:
diff_root = demo_path("04_review_diff_example")
remove_demo_path(diff_root)
try:
    diff_root.mkdir()
    diff_path = diff_root / "demo.ipynb"
    write_nb(new_nb([
        mk_cell("value = 1\nvalue", cell_type="code"),
        mk_cell("Original note", cell_type="markdown"),
    ]), diff_path)
    subprocess.run(["git", "init"], cwd=diff_root, check=True, capture_output=True)
    subprocess.run(["git", "add", "demo.ipynb"], cwd=diff_root, check=True, capture_output=True)
    subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=diff_root, check=True, capture_output=True)

    diff_nb_json = read_nb(diff_path)
    diff_nb_json.cells[0].source = "value = 2\nvalue"
    diff_nb_json.cells[1].source = "Updated note that will not appear in the code diff"
    write_nb(diff_nb_json, diff_path)

    diff_output = StringIO()
    with redirect_stdout(diff_output):
        diff_nb(str(diff_path), ref_a="HEAD")
    print(diff_output.getvalue())
finally:
    remove_demo_path(diff_root)